Must have spare memory, it takes a while and the installation of this model is around 15 gbs if fail prob low memory

In [1]:
import os

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch
from PIL import Image

c:\Users\Alex\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# img = Image.open(r"C:\Users\Alex\Music\project\COS40005-Computing-Technology-Project-A-H\demonstration2_unet\demo_input\TCGA_CS_4941_19960909_11.tif").convert("RGB")
# img = img.resize((512, 512))
# img.save("temp.png")

In [ ]:
#https://huggingface.co/lingshu-medical-mllm/Lingshu-7B
#Model import and processor setup from lingshu 
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "lingshu-medical-mllm/Lingshu-7B",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    #device_map="cuda", #Use if spare memory on GPU 
    cache_dir=r"A:\model"
    #low_cpu_mem_usage=True
)

processor = AutoProcessor.from_pretrained("lingshu-medical-mllm/Lingshu-7B")

c:\Users\Alex\AppData\Local\Programs\Python\Python312\Lib\site-packages\accelerate\utils\modeling.py:1566: UserWarning: Current model requires 3752 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 13.52it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You ca

In [4]:
#Message input
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                #"image": r"C:\Users\Alex\Music\project\COS40005-Computing-Technology-Project-A-H\demonstration2_unet\demo_input\TCGA_CS_4941_19960909_11.tif",
                "image": r"C:\Users\Alex\Music\project\COS40005-Computing-Technology-Project-A-H\demonstration2_unet\demo_input\temp.png",
            },
            {"type": "text", 
             "text": "Analyze this brain ct scan image and provide a detailed report on any abnormalities you find or if it doesnt contain any abnormalities."},
        ],
    }
]

In [ ]:
# Preparation for inference
text = processor.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

#convert image and text into pytorch tensors for model input
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)

In [6]:
# Convert input into tokens and decodes tokens into output text
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

: 

In [ ]:
print(output_text)

['The CT scan of the brain shows no evidence of acute intracranial hemorrhage, mass effect, midline shift, or significant cerebral edema. The ventricles and sulci appear normal in size and configuration, indicating no signs of hydrocephalus or atrophy. The bone structures of the skull are intact without fractures. Overall, the scan does not reveal any acute abnormalities.']
